In [2]:
import geopandas as gpd
from sqlalchemy import create_engine, text
import os


In [16]:
def load_vector_layer(db_name, user, password, host, port, table_name, schema='public', geom_col='geom'):
    """
    Connects to a PostGIS-enabled PostgreSQL database and loads a vector layer as a GeoDataFrame.
    
    Parameters:
    - db_name (str): Name of the PostgreSQL database.
    - user (str): Database username.
    - password (str): Database password.
    - host (str): Host address (e.g., 'localhost' or IP).
    - port (int): Port number (e.g., 5432).
    - table_name (str): Name of the table (vector layer) to load.
    - schema (str): Optional. Database schema containing the table (default is 'public').

    Returns:
    - gpd.GeoDataFrame: A GeoDataFrame containing the vector layer.
    """
    try:
        # Use pg8000 (pure Python driver)
        conn_str = f"postgresql+pg8000://{user}:{password}@{host}:{port}/{db_name}"
        engine = create_engine(conn_str)

        sql = text(f"SELECT * FROM {schema}.{table_name}")

        # Open a connection explicitly (SQLAlchemy 2.x requirement)
        with engine.connect() as conn:
            gdf = gpd.read_postgis(sql, conn, geom_col=geom_col)
        
        print(f"Successfully loaded {table_name} ({len(gdf)} features)")
        return gdf

    except Exception as e:
        print(f"Error loading vector layer: {e}")
        return None
    
def split_by_grid(gdf_input, gdf_grid):
    # Optional but VERY important for speed
    # (Shapely 2 / PyGEOS backend)
    gdf = gdf_input.copy()
    grid = gdf_grid.copy()

    gdf_geometry_name = gdf.geometry.name
    grid_geometry_name = grid.geometry.name

    # Make geometries valid (prevents topology errors & slowdowns)
    gdf[gdf_geometry_name] = gdf.make_valid()
    grid[grid_geometry_name] = grid.make_valid()

    # Keep only needed columns
    grid = grid[["tile_index", grid_geometry_name]]   # cell_id = your grid unique ID

    # ---- SPLIT coastline by grid (spatial-index accelerated) ----
    split = gpd.overlay(
        gdf,
        grid,
        how="intersection",
        keep_geom_type=True
    )

    # ---- DISSOLVE per grid cell ----
    result = split.dissolve(
        by="tile_index",
        as_index=False
    )

    return result

In [4]:
gdf_1d_grid = load_vector_layer(
    db_name='geoserver',
    user='geoserver',
    password='geoserver',
    host='192.168.250.233',
    port=5555,
    table_name='global_klab_1d_tiles',
    schema='klab_grids'
)

gdf_3d_grid = load_vector_layer(
    db_name='geoserver',
    user='geoserver',
    password='geoserver',
    host='192.168.250.233',
    port=5555,
    table_name='global_klab_3d_tiles',
    schema='klab_grids'
)


Successfully loaded global_klab_1d_tiles (50760 features)
Successfully loaded global_klab_3d_tiles (5640 features)


In [7]:
gdf_input = gpd.read_file(r"C:\Users\admin\Documents\01_Ruben_Scripts\turtle_lab\vector_management\global_osm_coastline_6km_buffer_grid_split_fixed_4326.shp")

In [24]:
gdf_input.head()

,t_indx_1d,geometry,t_indx_3d,gid
0,11992.0,"POLYGON ((-68.76867 -56.50664, -68.76923 -56.5...",1358.0,1
1,11993.0,"POLYGON ((-67.25628 -56.00524, -67.25723 -56.0...",1358.0,1
2,12032.0,"POLYGON ((-28.13001 -56.65925, -28.12868 -56.6...",1371.0,3
3,12033.0,"MULTIPOLYGON (((-27.23699 -56.75594, -27.23719...",1371.0,3
4,12349.0,"POLYGON ((-71.00037 -55.12132, -71.00067 -55.1...",1357.0,5


In [ ]:
gdf_input = gdf_input.drop(columns=["tile_index", "t_indx_1d"])

In [23]:
gdf_input = gdf_input.rename(columns={"tile_index": "t_indx_1d"})

In [ ]:
gdf_input = split_by_grid(gdf_input, gdf_1d_grid)
# Save result


In [25]:
gdf_input.to_file("global_osm_coastline_6km_buffer_grid_split_fixed_real_4326.shp", driver="ESRI Shapefile")